# LFW Grad-CAM — 06. Representative case visualization

모든 집단 분석이 끝난 뒤에만 stable/high-error/rank-flip/
threshold-crossing 예시를 선택합니다. 이 단계는 이미 저장된 heatmap을
읽어 그림만 만들며 Grad-CAM을 다시 생성하지 않습니다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

EXECUTION = CONFIG["execution"]
MODEL_PROFILE = str(EXECUTION["model_profile"])
MODE = str(EXECUTION["mode"])
DATA_FRACTION = float(EXECUTION["data_fraction"])
SEED = int(EXECUTION["seed"])
EXECUTE_STAGE = bool(EXECUTION["execute_stage"])
WRITE_OUTPUTS = bool(EXECUTION["write_outputs"])
OVERWRITE = bool(EXECUTION["overwrite"])

if MODEL_PROFILE not in CONFIG["models"]["selected_profiles"]:
    raise ValueError(f"선택되지 않은 모델 profile: {MODEL_PROFILE}")
MODEL_NAME = str(CONFIG["models"]["profiles"][MODEL_PROFILE]["family"])
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [ ]:
from research.runtime import RunStore, resolve_active_dataset_run

STEP2_RUN_DIR = resolve_active_dataset_run(
    PROJECT_ROOT / CONFIG["run"]["root"],
    dataset_id=str(CONFIG["run"]["dataset_id"]),
    directory_template=str(CONFIG["run"]["dataset_date_dir_template"]),
)
RUN = RunStore.open(STEP2_RUN_DIR)
WORKFLOW_ROOT = (
    STEP2_RUN_DIR / CONFIG["workflow"]["artifact_subdir"]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from research.explainability.gradcam import (
    read_population_heatmaps,
    select_population_representative_cases,
)

JOINED_METRICS_PATH = WORKFLOW_ROOT / CONFIG["workflow"]["joined_metrics_path"]
SALIENCY_ARTIFACT_DIR = WORKFLOW_ROOT / CONFIG["workflow"]["saliency_population_dir"]
SELECTED_MANIFEST_PATH = WORKFLOW_ROOT / CONFIG["workflow"]["selected_manifest_path"]
ALIGNED_FACES_NPY_PATH = PROJECT_ROOT / CONFIG["aligned_crops"]["faces_path"]
CASE_MANIFEST_OUTPUT_PATH = WORKFLOW_ROOT / CONFIG["workflow"]["representative_cases_path"]
FIGURE_OUTPUT_DIR = WORKFLOW_ROOT / CONFIG["workflow"]["figure_dir"]
CASES_PER_GROUP = int(
    CONFIG["gradcam"]["representative_case_visualization"][
        "samples_per_stratum"
    ]
)
VISUALIZATION_THRESHOLD_POLICY = str(
    CONFIG["gradcam"]["representative_case_visualization"]["threshold_policy"]
)


In [ ]:
if EXECUTE_STAGE:
    required = {
        "joined": JOINED_METRICS_PATH,
        "saliency": SALIENCY_ARTIFACT_DIR,
        "selected": SELECTED_MANIFEST_PATH,
        "aligned_faces": ALIGNED_FACES_NPY_PATH,
    }
    missing = [name for name, value in required.items() if value is None]
    if missing:
        raise RuntimeError(f"입력 경로가 비어 있습니다: {missing}")
    joined = pd.read_csv(required["joined"], low_memory=False)
    if "threshold_policy" not in joined:
        raise ValueError("joined metrics에 threshold_policy가 없습니다.")
    available_policies = set(joined["threshold_policy"].dropna().astype(str))
    if VISUALIZATION_THRESHOLD_POLICY not in available_policies:
        raise ValueError(
            "대표 사례 정책이 joined metrics에 없습니다: "
            f"{VISUALIZATION_THRESHOLD_POLICY}; available={sorted(available_policies)}"
        )
    joined = joined.loc[
        joined["threshold_policy"].astype(str)
        == VISUALIZATION_THRESHOLD_POLICY
    ].copy()
    cases = select_population_representative_cases(
        joined,
        cases_per_group=CASES_PER_GROUP,
        seed=SEED,
    )
    heatmap_ids, heatmaps = read_population_heatmaps(
        required["saliency"]
    )
    heatmap_index = {
        str(sample_id): index
        for index, sample_id in enumerate(heatmap_ids.astype(str))
    }
    selected = pd.read_csv(required["selected"])
    selected["sample_id"] = selected["sample_id"].astype(str)
    aligned = np.load(
        required["aligned_faces"],
        mmap_mode="r",
        allow_pickle=False,
    )
    case_summary = {
        "case_count": int(len(cases)),
        "case_groups": cases["case_group"].value_counts().to_dict(),
        "threshold_policy": VISUALIZATION_THRESHOLD_POLICY,
        "regenerated_gradcam": False,
    }
    if WRITE_OUTPUTS:
        if CASE_MANIFEST_OUTPUT_PATH is None or FIGURE_OUTPUT_DIR is None:
            raise RuntimeError("case manifest와 figure 출력 경로를 지정하세요.")
        case_path = Path(CASE_MANIFEST_OUTPUT_PATH).resolve()
        figure_dir = Path(FIGURE_OUTPUT_DIR).resolve()
        if (case_path.exists() or figure_dir.exists()) and not OVERWRITE:
            raise FileExistsError("기존 case 결과를 덮어쓸 수 없습니다.")
        case_path.parent.mkdir(parents=True, exist_ok=True)
        figure_dir.mkdir(parents=True, exist_ok=True)
        cases.to_csv(case_path, index=False, encoding="utf-8")
        sample_to_face = selected.set_index("sample_id")[
            "aligned_face_index"
        ].astype(int).to_dict()
        for row in cases.itertuples(index=False):
            sample_id = str(row.sample_id)
            image = np.asarray(aligned[sample_to_face[sample_id]])
            heatmap = heatmaps[heatmap_index[sample_id]]
            figure, axis = plt.subplots(figsize=(3, 3))
            axis.imshow(image)
            axis.imshow(
                heatmap,
                cmap="jet",
                alpha=0.45,
                extent=(0, image.shape[1], image.shape[0], 0),
            )
            axis.set_title(f"{row.case_group}: {sample_id}")
            axis.axis("off")
            figure.savefig(
                figure_dir / f"{row.case_id}.png",
                dpi=160,
                bbox_inches="tight",
            )
            plt.close(figure)
    if WRITE_OUTPUTS:
        RUN.complete()
else:
    cases = pd.DataFrame()
    case_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
display(cases.head(20))
case_summary


사례 그림은 집단 통계의 보조 설명입니다. 사례에서 보인 패턴을 전체
이미지의 일반적 원인으로 확대 해석하지 않습니다.
